In [ ]:
# Task 5: Strategy Backtesting
# Validate optimized portfolio against benchmark

import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import pickle
import json

# Import custom modules
from backtester import PortfolioBacktester
from performance_analyzer import PerformanceAnalyzer
from portfolio_optimizer import PortfolioOptimizer

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Load configurations
with open('../config/backtest_config.yaml', 'r') as f:
    backtest_config = yaml.safe_load(f)

print("Configuration loaded successfully.")

### Load Data and Portfolio Weights

In [ ]:
print("=" * 80)
print("TASK 5: STRATEGY BACKTESTING")
print("=" * 80)

print("\n1. LOADING DATA AND PORTFOLIO WEIGHTS...")

# Load price data
prices_path = '../data/processed/adjusted_close_prices.csv'
prices = pd.read_csv(prices_path, index_col=0, parse_dates=True)

print(f"Loaded price data: {prices.shape}")
print(f"Date range: {prices.index[0]} to {prices.index[-1]}")
print(f"Assets: {list(prices.columns)}")

# Load optimized portfolio weights from Task 4
weights_path = '../models/portfolio_weights.pkl'
with open(weights_path, 'rb') as f:
    portfolio_weights = pickle.load(f)

print(f"\nLoaded portfolio weights from Task 4:")
print(f"Assets: {portfolio_weights['assets']}")
print(f"Generation date: {portfolio_weights['generation_date']}")

# Extract strategy weights (Max Sharpe portfolio)
strategy_weights = dict(zip(portfolio_weights['assets'], portfolio_weights['max_sharpe']))

print(f"\nStrategy Portfolio Weights (Max Sharpe):")
for asset, weight in strategy_weights.items():
    if weight > 0.001:
        print(f"  {asset}: {weight:.1%}")

# Define benchmark weights (60% SPY, 40% BND)
benchmark_weights = backtest_config['benchmark']['weights']

print(f"\nBenchmark Portfolio Weights:")
for asset, weight in benchmark_weights.items():
    print(f"  {asset}: {weight:.1%}")

# Verify all assets exist in price data
for asset in list(strategy_weights.keys()) + list(benchmark_weights.keys()):
    if asset not in prices.columns:
        print(f"Warning: Asset {asset} not in price data")

### Initialize Backtester

In [ ]:
print("\n" + "=" * 80)
print("2. INITIALIZING BACKTESTER")
print("=" * 80)

# Initialize backtester
initial_capital = backtest_config['backtest']['initial_capital']
risk_free_rate = backtest_config['metrics']['risk_free_rate']
transaction_cost = backtest_config['backtest']['transaction_cost']
slippage = backtest_config['backtest']['slippage']

backtester = PortfolioBacktester(
    prices=prices,
    initial_capital=initial_capital,
    risk_free_rate=risk_free_rate,
    transaction_cost=transaction_cost,
    slippage=slippage
)

print(f"Backtester initialized with:")
print(f"  Initial capital: ${initial_capital:,.0f}")
print(f"  Risk-free rate: {risk_free_rate:.2%}")
print(f"  Transaction cost: {transaction_cost:.1%}")
print(f"  Slippage: {slippage:.1%}")

# Set backtest period
start_date = backtest_config['backtest']['start_date']
end_date = backtest_config['backtest']['end_date']
rebalancing_freq = backtest_config['strategy']['rebalancing_frequency']
rebalancing_threshold = backtest_config['strategy']['rebalancing_threshold']

print(f"\nBacktest Parameters:")
print(f"  Period: {start_date} to {end_date}")
print(f"  Rebalancing frequency: {rebalancing_freq}")
print(f"  Rebalancing threshold: {rebalancing_threshold:.0%}")

### Run Backtest

In [ ]:
print("\n" + "=" * 80)
print("3. RUNNING BACKTEST")
print("=" * 80)

# Run backtest
backtest_results = backtester.run_backtest(
    strategy_weights=strategy_weights,
    benchmark_weights=benchmark_weights,
    start_date=start_date,
    end_date=end_date,
    rebalancing_freq=rebalancing_freq,
    rebalancing_threshold=rebalancing_threshold
)

print("\nBacktest completed successfully!")

# Display key results
strategy_metrics = backtest_results['strategy']['metrics']
benchmark_metrics = backtest_results['benchmark']['metrics']
relative_metrics = backtest_results['relative']

print(f"\nKEY RESULTS:")
print("-" * 60)
print(f"Strategy Final Value: ${strategy_metrics['final_value']:,.0f}")
print(f"Benchmark Final Value: ${benchmark_metrics['final_value']:,.0f}")
print(f"Excess Return: {relative_metrics['excess_return']:.1f}%")
print(f"Alpha: {relative_metrics['alpha']:.2%}")
print(f"Information Ratio: {relative_metrics['information_ratio']:.3f}")

### Visualize Results

In [ ]:
print("\n" + "=" * 80)
print("4. VISUALIZING BACKTEST RESULTS")
print("=" * 80)

# Create output directory
os.makedirs('../figures/task5', exist_ok=True)

# 4.1 Plot cumulative returns
print("\n4.1 Plotting cumulative returns...")
cumulative_fig_path = '../figures/task5/cumulative_returns.png'
backtester.plot_cumulative_returns(save_path=cumulative_fig_path)

# 4.2 Plot drawdown comparison
print("\n4.2 Plotting drawdown comparison...")
drawdown_fig_path = '../figures/task5/drawdown_comparison.png'
backtester.plot_drawdown(save_path=drawdown_fig_path)

# 4.3 Plot rolling metrics
print("\n4.3 Plotting rolling metrics...")
rolling_window = backtest_config['visualization']['rolling_window']
rolling_fig_path = '../figures/task5/rolling_metrics.png'
backtester.plot_rolling_metrics(window=rolling_window, save_path=rolling_fig_path)

# 4.4 Plot performance summary
print("\n4.4 Plotting performance summary...")
summary_fig_path = '../figures/task5/performance_summary.png'
backtester.plot_performance_summary(save_path=summary_fig_path)

print("\nAll visualizations saved to '../figures/task5/'")

### Calculate Advanced Performance Metrics

In [ ]:
print("\n" + "=" * 80)
print("5. ADVANCED PERFORMANCE ANALYSIS")
print("=" * 80)

# Extract returns for analysis
strategy_returns = backtest_results['strategy']['results']['returns'][1:]  # Exclude first day
benchmark_returns = backtest_results['benchmark']['results']['returns'][1:]

print(f"\nAnalyzing {len(strategy_returns)} daily returns...")

# Calculate advanced metrics for strategy
strategy_advanced_metrics = PerformanceAnalyzer.calculate_all_advanced_metrics(
    returns=strategy_returns,
    benchmark_returns=benchmark_returns,
    risk_free_rate=risk_free_rate
)

print("\nADVANCED STRATEGY METRICS:")
print("-" * 40)

# Display key advanced metrics
advanced_metrics_to_display = {
    'ulcer_index': 'Ulcer Index',
    'tail_ratio_95': 'Tail Ratio (95%)',
    'pain_index': 'Pain Index',
    'omega_ratio': 'Omega Ratio',
    'mar_ratio': 'MAR Ratio',
    'gain_to_pain_ratio': 'Gain to Pain Ratio',
    'hurst_exponent': 'Hurst Exponent',
    'skewness': 'Skewness',
    'kurtosis': 'Kurtosis'
}

for metric_key, display_name in advanced_metrics_to_display.items():
    if metric_key in strategy_advanced_metrics:
        value = strategy_advanced_metrics[metric_key]
        print(f"{display_name:<20}: {value:.4f}")

# Display statistical test results
print("\nSTATISTICAL TESTS (Strategy vs Benchmark):")
print("-" * 40)

if 'wilcoxon_p_value' in strategy_advanced_metrics:
    p_value = strategy_advanced_metrics['wilcoxon_p_value']
    significant = strategy_advanced_metrics['wilcoxon_significant_at_5%']
    
    print(f"Wilcoxon Test p-value: {p_value:.6f}")
    print(f"Significant at 5% level: {'Yes' if significant else 'No'}")
    
    if 'wilcoxon_effect_size' in strategy_advanced_metrics:
        effect_size = strategy_advanced_metrics['wilcoxon_effect_size']
        print(f"Effect Size: {effect_size:.4f}")

# Bootstrap test
print("\nBOOTSTRAP ANALYSIS:")
print("-" * 40)

bootstrap_results = PerformanceAnalyzer.perform_bootstrap_test(
    strategy_returns, benchmark_returns, n_bootstrap=1000
)

print(f"Mean Difference: {bootstrap_results['original_mean_diff']*100:.4f}%")
print(f"95% Confidence Interval: [{bootstrap_results['confidence_interval_lower']*100:.4f}%, "
      f"{bootstrap_results['confidence_interval_upper']*100:.4f}%]")
print(f"Bootstrap p-value: {bootstrap_results['p_value']:.6f}")
print(f"Significant: {'Yes' if bootstrap_results['significant'] else 'No'}")

# Regime analysis
print("\nREGIME ANALYSIS:")
print("-" * 40)

regime_results = PerformanceAnalyzer.perform_regime_analysis(strategy_returns - benchmark_returns)

if 'regime_analysis' in strategy_advanced_metrics:
    regime_returns = strategy_advanced_metrics['regime_analysis']['regime_returns']
    
    for regime, metrics in regime_returns.items():
        print(f"\n{regime.replace('_', ' ').title()} Regime:")
        print(f"  Mean Return: {metrics['mean_return']*100:.4f}%")
        print(f"  Sharpe Ratio: {metrics['sharpe_ratio']:.4f}")
        print(f"  Observations: {metrics['count']}")

### Backtest Robustness Analysis

In [ ]:
print("\n" + "=" * 80)
print("6. ROBUSTNESS ANALYSIS")
print("=" * 80)

print("\nRunning robustness checks with different parameters...")

# Test different rebalancing frequencies
rebalancing_options = ['monthly', 'quarterly', 'none']
robustness_results = {}

for freq in rebalancing_options:
    print(f"\nTesting with {freq} rebalancing...")
    
    try:
        # Run backtest with different frequency
        robustness_backtest = backtester.run_backtest(
            strategy_weights=strategy_weights,
            benchmark_weights=benchmark_weights,
            start_date=start_date,
            end_date=end_date,
            rebalancing_freq=freq,
            rebalancing_threshold=rebalancing_threshold
        )
        
        strategy_final = robustness_backtest['strategy']['metrics']['final_value']
        benchmark_final = robustness_backtest['benchmark']['metrics']['final_value']
        excess = ((strategy_final / benchmark_final) - 1) * 100
        
        robustness_results[freq] = {
            'strategy_final': strategy_final,
            'excess_return': excess,
            'sharpe_ratio': robustness_backtest['strategy']['metrics']['sharpe_ratio']
        }
        
        print(f"  Strategy Final: ${strategy_final:,.0f}")
        print(f"  Excess Return: {excess:.1f}%")
        print(f"  Sharpe Ratio: {robustness_results[freq]['sharpe_ratio']:.3f}")
        
    except Exception as e:
        print(f"  Failed: {str(e)}")
        robustness_results[freq] = None

# Test different transaction cost assumptions
print("\n\nTRANSACTION COST SENSITIVITY:")
print("-" * 40)

cost_scenarios = [0.000, 0.001, 0.002, 0.005]  # 0%, 0.1%, 0.2%, 0.5%

for cost in cost_scenarios:
    # Create new backtester with different transaction cost
    sensitivity_backtester = PortfolioBacktester(
        prices=prices,
        initial_capital=initial_capital,
        risk_free_rate=risk_free_rate,
        transaction_cost=cost,
        slippage=slippage
    )
    
    sensitivity_results = sensitivity_backtester.run_backtest(
        strategy_weights=strategy_weights,
        benchmark_weights=benchmark_weights,
        start_date=start_date,
        end_date=end_date,
        rebalancing_freq=rebalancing_freq,
        rebalancing_threshold=rebalancing_threshold
    )
    
    strategy_final = sensitivity_results['strategy']['metrics']['final_value']
    transaction_costs = sensitivity_results['strategy']['results']['total_transaction_costs']
    
    print(f"Transaction Cost {cost:.1%}:")
    print(f"  Strategy Final: ${strategy_final:,.0f}")
    print(f"  Total Costs: ${transaction_costs:,.0f}")
    print(f"  Cost as % of Final: {transaction_costs/strategy_final:.2%}")

### Save Backtest Results

In [ ]:
print("\n" + "=" * 80)
print("7. SAVING BACKTEST RESULTS")
print("=" * 80)

# Save backtest results
results_path = '../models/backtest_results.pkl'
backtester.save_backtest_results(results_path)

print(f"Backtest results saved to: {results_path}")

# Save performance metrics CSV
metrics_data = []

for metric_name in ['total_return', 'annualized_return', 'sharpe_ratio', 
                    'sortino_ratio', 'max_drawdown', 'calmar_ratio', 
                    'value_at_risk_95', 'win_rate']:
    
    metrics_data.append({
        'metric': metric_name.replace('_', ' ').title(),
        'strategy': strategy_metrics.get(metric_name, 0),
        'benchmark': benchmark_metrics.get(metric_name, 0),
        'difference': strategy_metrics.get(metric_name, 0) - benchmark_metrics.get(metric_name, 0)
    })

metrics_df = pd.DataFrame(metrics_data)
metrics_path = '../data/processed/backtest_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)

print(f"Performance metrics saved to: {metrics_path}")

### Conclusions and Reflections

In [ ]:
print("\n" + "=" * 80)
print("8. CONCLUSIONS AND REFLECTIONS")
print("=" * 80)

# Determine overall conclusion
excess_return = relative_metrics['excess_return']
alpha = relative_metrics['alpha']
information_ratio = relative_metrics['information_ratio']
significant = strategy_advanced_metrics.get('wilcoxon_significant_at_5%', False)

print("\nOVERALL ASSESSMENT:")
print("-" * 60)

if excess_return > 0 and alpha > 0 and significant:
    print("✅ STRONG VALIDATION")
    print("   • Strategy outperforms benchmark with statistical significance")
    print("   • Positive alpha indicates skill-based outperformance")
    print("   • Robust across different rebalancing frequencies")
    
    if information_ratio > 0.5:
        print("   • Strong information ratio indicates consistent outperformance")
    
    print("\nBUSINESS IMPLICATIONS:")
    print("   • Strategy adds value over traditional 60/40 portfolio")
    print("   • Validates forecast-integrated portfolio optimization approach")
    print("   • Provides empirical evidence for investment committee approval")
    
elif excess_return > 0:
    print("⚠️ MODERATE VALIDATION")
    print("   • Strategy outperforms but without statistical significance")
    print("   • May need longer backtest period or different market conditions")
    print("   • Consider as promising but requiring further validation")
    
    print("\nBUSINESS IMPLICATIONS:")
    print("   • Strategy shows potential but not conclusively proven")
    print("   • Consider pilot implementation with limited capital")
    print("   • Monitor closely and re-evaluate in 6-12 months")
    
else:
    print("❌ LIMITED VALIDATION")
    print("   • Strategy failed to outperform benchmark")
    print("   • May indicate issues with forecast accuracy or optimization")
    print("   • Requires significant refinement before implementation")
    
    print("\nBUSINESS IMPLICATIONS:")
    print("   • Do not implement in current form")
    print("   • Review forecast models and optimization assumptions")
    print("   • Consider alternative asset combinations or constraints")

print("\nLIMITATIONS OF THIS BACKTEST:")
print("-" * 60)
print("1. SINGLE HISTORICAL PERIOD: Backtest covers only one year")
print("2. ASSUMPTION-DEPENDENT: Results sensitive to transaction cost and slippage assumptions")
print("3. FORECAST BIAS: Uses forecast from Task 3 which has its own limitations")
print("4. SIMPLIFIED EXECUTION: Assumes perfect liquidity and no market impact")
print("5. REGIME DEPENDENCE: Performance may vary across different market conditions")

print("\nRECOMMENDATIONS FOR FUTURE WORK:")
print("-" * 60)
print("1. EXTEND BACKTEST PERIOD: Test strategy across multiple market cycles")
print("2. INCORPORATE MORE ASSETS: Expand beyond TSLA, BND, SPY for better diversification")
print("3. ADD ALTERNATIVE BENCHMARKS: Compare against more sophisticated benchmarks")
print("4. IMPLEMENT WALK-FORWARD OPTIMIZATION: Update portfolio weights periodically")
print("5. INCLUDE STRESS TESTING: Test performance during market crises")
print("6. ADD SENSITIVITY ANALYSIS: Vary all parameters systematically")

print("\nIMMEDIATE NEXT STEPS:")
print("-" * 60)
print("1. PRESENT FINDINGS: Share results with investment committee")
print("2. DECIDE ON IMPLEMENTATION: Based on validation results")
print("3. SET UP MONITORING: Establish KPIs and review process")
print("4. PLAN PILOT PROGRAM: If approved, design controlled implementation")
print("5. DOCUMENT LESSONS LEARNED: Capture insights for future projects")